In [ ]:
!pip install -q flask boto3 "moto[dynamodb]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.3/90.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 85.4 MB/s eta 0:00:00


In [ ]:
import os, re, json, time, uuid, base64, random, threading
from datetime import datetime, date, timezone
from decimal import Decimal
import boto3
from boto3.dynamodb.conditions import Key, Attr
from botocore.exceptions import ClientError

MODE = 'mock'                    # 'mock'  or  'aws'
LISTS_TABLE, TASKS_TABLE = 'TodoLists', 'TodoTasks'

if MODE == 'mock':
    from moto import mock_aws
    os.environ['AWS_ACCESS_KEY_ID'] = 'testing'
    os.environ['AWS_SECRET_ACCESS_KEY'] = 'testing'
    REGION = 'us-east-1'
    _mock = mock_aws(); _mock.start()
    aws_kwargs = {'region_name': REGION}
else:
    from google.colab import userdata
    REGION = userdata.get('AWS_REGION')
    aws_kwargs = dict(region_name=REGION,
                      aws_access_key_id=userdata.get('AWS_ACCESS_KEY_ID'),
                      aws_secret_access_key=userdata.get('AWS_SECRET_ACCESS_KEY'))

ddb = boto3.resource('dynamodb', **aws_kwargs)

def ensure_tables():
    existing = [t.name for t in ddb.tables.all()]
    if LISTS_TABLE not in existing:
        ddb.create_table(TableName=LISTS_TABLE,
                         KeySchema=[{'AttributeName': 'list_id', 'KeyType': 'HASH'}],
                         AttributeDefinitions=[{'AttributeName': 'list_id', 'AttributeType': 'S'}],
                         BillingMode='PAY_PER_REQUEST').wait_until_exists()
    if TASKS_TABLE not in existing:
        # GSI on list_id = fast "get all tasks of a list" query
        ddb.create_table(TableName=TASKS_TABLE,
                         KeySchema=[{'AttributeName': 'task_id', 'KeyType': 'HASH'}],
                         AttributeDefinitions=[{'AttributeName': 'task_id', 'AttributeType': 'S'},
                                               {'AttributeName': 'list_id', 'AttributeType': 'S'}],
                         GlobalSecondaryIndexes=[{'IndexName': 'list_id-index',
                                                  'KeySchema': [{'AttributeName': 'list_id', 'KeyType': 'HASH'}],
                                                  'Projection': {'ProjectionType': 'ALL'}}],
                         BillingMode='PAY_PER_REQUEST').wait_until_exists()
    return ddb.Table(LISTS_TABLE), ddb.Table(TASKS_TABLE)

lists_table, tasks_table = ensure_tables()
print(f'Mode: {MODE} | Region: {REGION} | Tables: {LISTS_TABLE}, {TASKS_TABLE}')

Mode: mock | Region: us-east-1 | Tables: TodoLists, TodoTasks


In [ ]:
VALID_PRIORITY = {'Low': 2, 'Medium': 1, 'High': 0}
CODE_CHARS = 'ABCDEFGHJKLMNPQRSTUVWXYZ23456789'      # no confusing characters (0/O, 1/I)
ONLINE_SECONDS = 10

def _json_default(o):
    return int(o) if isinstance(o, Decimal) else str(o)

def response(code, payload):
    return {'statusCode': code, 'headers': {'Content-Type': 'application/json'},
            'body': json.dumps(payload, default=_json_default)}

def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def clean_user(u):
    u = (u or '').strip()
    return u if re.fullmatch(r'[A-Za-z0-9_]{2,20}', u) else None

def scan_all(tbl, **kw):
    resp = tbl.scan(**kw)
    items = resp['Items']
    while 'LastEvaluatedKey' in resp:
        resp = tbl.scan(ExclusiveStartKey=resp['LastEvaluatedKey'], **kw)
        items += resp['Items']
    return items

def tasks_of(list_id):
    kw = dict(IndexName='list_id-index', KeyConditionExpression=Key('list_id').eq(list_id))
    resp = tasks_table.query(**kw)
    items = resp['Items']
    while 'LastEvaluatedKey' in resp:
        resp = tasks_table.query(ExclusiveStartKey=resp['LastEvaluatedKey'], **kw)
        items += resp['Items']
    # open tasks first, then High -> Low priority, then oldest first
    items.sort(key=lambda t: (t.get('done', False), VALID_PRIORITY.get(t.get('priority'), 1), t['created_at']))
    return items

def touch_list(list_id, user, message):
    # bump the list version (this is what tells other users "something changed") and log the activity
    entry = {'t': now_iso(), 'user': user, 'msg': message}
    resp = lists_table.update_item(
        Key={'list_id': list_id},
        UpdateExpression='SET updated_at = :t, activity = list_append(activity, :a) ADD #ver :one',
        ExpressionAttributeNames={'#ver': 'version'},
        ExpressionAttributeValues={':t': now_iso(), ':a': [entry], ':one': 1},
        ReturnValues='ALL_NEW')
    attrs = resp['Attributes']
    if len(attrs['activity']) > 80:                         # keep the feed short
        lists_table.update_item(Key={'list_id': list_id}, UpdateExpression='SET activity = :x',
                                ExpressionAttributeValues={':x': attrs['activity'][-50:]})
    return int(attrs['version'])

def member_list(body):
    user = clean_user(body.get('user'))
    if not user:
        return None, None, response(400, {'error': 'Enter a name (2-20 letters, numbers or _).'})
    item = lists_table.get_item(Key={'list_id': body.get('list_id', '')}, ConsistentRead=True).get('Item')
    if not item:
        return None, None, response(404, {'error': 'List not found (it may have been deleted).'})
    if user not in item['members']:
        return None, None, response(403, {'error': 'You are not a member of this list.'})
    return user, item, None

def public_list(item):
    return {'list_id': item['list_id'], 'title': item['title'], 'share_code': item['share_code'],
            'owner': item['owner'], 'members': item['members'], 'version': int(item['version'])}

# ---------------- lists ----------------
def handle_list_create(body):
    user = clean_user(body.get('user'))
    title = (body.get('title') or '').strip()[:60]
    if not user: return response(400, {'error': 'Enter a name (2-20 letters, numbers or _).'})
    if not title: return response(400, {'error': 'Enter a list title.'})
    used = {i['share_code'] for i in scan_all(lists_table, ProjectionExpression='share_code')}
    code = ''.join(random.choices(CODE_CHARS, k=6))
    while code in used:
        code = ''.join(random.choices(CODE_CHARS, k=6))
    item = {'list_id': uuid.uuid4().hex[:10], 'title': title, 'share_code': code, 'owner': user,
            'members': [user], 'presence': {}, 'activity': [], 'version': 1,
            'created_at': now_iso(), 'updated_at': now_iso()}
    lists_table.put_item(Item=item)
    touch_list(item['list_id'], user, f'created the list "{title}"')
    return response(200, public_list(item))

def handle_list_join(body):
    user = clean_user(body.get('user'))
    code = (body.get('code') or '').strip().upper()
    if not user: return response(400, {'error': 'Enter a name (2-20 letters, numbers or _).'})
    found = scan_all(lists_table, FilterExpression=Attr('share_code').eq(code))
    if not code or not found:
        return response(404, {'error': 'No list found with that share code.'})
    item = found[0]
    if user not in item['members']:
        lists_table.update_item(Key={'list_id': item['list_id']},
                                UpdateExpression='SET members = list_append(members, :u)',
                                ExpressionAttributeValues={':u': [user]})
        touch_list(item['list_id'], user, 'joined the list')
        item = lists_table.get_item(Key={'list_id': item['list_id']}, ConsistentRead=True)['Item']
    return response(200, public_list(item))

def handle_list_mine(body):
    user = clean_user(body.get('user'))
    if not user: return response(400, {'error': 'Enter a name.'})
    items = scan_all(lists_table, FilterExpression=Attr('members').contains(user))
    items.sort(key=lambda i: i['created_at'])
    return response(200, {'lists': [public_list(i) for i in items]})

def handle_list_delete(body):
    user, item, err = member_list(body)
    if err: return err
    if item['owner'] != user:
        return response(403, {'error': 'Only the owner can delete the list.'})
    with tasks_table.batch_writer() as batch:
        for t in tasks_of(item['list_id']):
            batch.delete_item(Key={'task_id': t['task_id']})
    lists_table.delete_item(Key={'list_id': item['list_id']})
    return response(200, {'deleted': item['list_id']})

# ---------------- tasks ----------------
def valid_due(s):
    try:
        date.fromisoformat(s); return True
    except ValueError:
        return False

def handle_task_add(body):
    user, lst, err = member_list(body)
    if err: return err
    title = (body.get('title') or '').strip()[:120]
    if not title: return response(400, {'error': 'Enter a task title.'})
    priority = body.get('priority') if body.get('priority') in VALID_PRIORITY else 'Medium'
    task = {'task_id': uuid.uuid4().hex[:12], 'list_id': lst['list_id'], 'title': title, 'done': False,
            'priority': priority, 'created_by': user, 'updated_by': user, 'version': 1,
            'created_at': now_iso(), 'updated_at': now_iso()}
    if body.get('assignee') in lst['members']:
        task['assignee'] = body['assignee']
    if body.get('due_date') and valid_due(body['due_date']):
        task['due_date'] = body['due_date']
    tasks_table.put_item(Item=task)
    touch_list(lst['list_id'], user, f'added "{title}"')
    return response(200, task)

def handle_task_update(body):
    user, lst, err = member_list(body)
    if err: return err
    task_id = body.get('task_id', '')
    sets, removes, names, values, changes = [], [], {'#ver': 'version'}, {':one': 1}, []

    if 'title' in body:
        t = (body['title'] or '').strip()[:120]
        if not t: return response(400, {'error': 'Title cannot be empty.'})
        sets.append('#title = :title'); names['#title'] = 'title'; values[':title'] = t; changes.append('renamed')
    if 'done' in body:
        sets.append('#done = :done'); names['#done'] = 'done'; values[':done'] = bool(body['done'])
        changes.append('completed' if body['done'] else 'reopened')
    if 'priority' in body and body['priority'] in VALID_PRIORITY:
        sets.append('#pr = :pr'); names['#pr'] = 'priority'; values[':pr'] = body['priority']
        changes.append(f"set priority {body['priority']} on")
    if 'assignee' in body:
        if body['assignee'] in lst['members']:
            sets.append('#as = :as'); names['#as'] = 'assignee'; values[':as'] = body['assignee']
            changes.append(f"assigned to {body['assignee']}:")
        else:
            removes.append('#as'); names['#as'] = 'assignee'; changes.append('unassigned')
    if 'due_date' in body:
        if body['due_date'] and valid_due(body['due_date']):
            sets.append('#due = :due'); names['#due'] = 'due_date'; values[':due'] = body['due_date']
            changes.append('changed due date of')
        elif not body['due_date']:
            removes.append('#due'); names['#due'] = 'due_date'; changes.append('cleared due date of')
    if not sets and not removes:
        return response(400, {'error': 'Nothing to update.'})

    sets += ['#ub = :ub', '#ua = :ua']
    names.update({'#ub': 'updated_by', '#ua': 'updated_at', '#lid': 'list_id'})
    values.update({':ub': user, ':ua': now_iso(), ':v': int(body.get('version', 0)), ':l': lst['list_id']})
    expr = 'SET ' + ', '.join(sets) + (' REMOVE ' + ', '.join(removes) if removes else '') + ' ADD #ver :one'

    try:
        resp = tasks_table.update_item(
            Key={'task_id': task_id}, UpdateExpression=expr,
            ConditionExpression='attribute_exists(task_id) AND #ver = :v AND #lid = :l',
            ExpressionAttributeNames=names, ExpressionAttributeValues=values, ReturnValues='ALL_NEW')
    except ClientError as e:
        if e.response['Error']['Code'] != 'ConditionalCheckFailedException':
            raise
        latest = tasks_table.get_item(Key={'task_id': task_id}, ConsistentRead=True).get('Item')
        if not latest or latest['list_id'] != lst['list_id']:
            return response(404, {'error': 'This task was deleted by someone else.'})
        return response(409, {'error': 'Someone else changed this task first. Showing the latest version.',
                              'task': latest})
    task = resp['Attributes']
    touch_list(lst['list_id'], user, f'{" & ".join(changes)} "{task["title"]}"'.replace(':"', ': "'))
    return response(200, task)

def handle_task_delete(body):
    user, lst, err = member_list(body)
    if err: return err
    task = tasks_table.get_item(Key={'task_id': body.get('task_id', '')}).get('Item')
    if not task or task['list_id'] != lst['list_id']:
        return response(404, {'error': 'Task not found.'})
    tasks_table.delete_item(Key={'task_id': task['task_id']})
    touch_list(lst['list_id'], user, f'deleted "{task["title"]}"')
    return response(200, {'deleted': task['task_id']})

# ---------------- real-time sync ----------------
def handle_sync(body):
    user, lst, err = member_list(body)
    if err: return err
    now = int(time.time())
    lists_table.update_item(Key={'list_id': lst['list_id']},
                            UpdateExpression='SET presence.#u = :ts',
                            ExpressionAttributeNames={'#u': user}, ExpressionAttributeValues={':ts': now})
    presence = {k: int(v) for k, v in lst.get('presence', {}).items()}
    presence[user] = now
    online = sorted(u for u, ts in presence.items() if now - ts <= ONLINE_SECONDS)

    version = int(lst['version'])
    if int(body.get('since', -1)) == version:
        return response(200, {'changed': False, 'version': version, 'online': online})
    return response(200, {'changed': True, 'version': version, 'online': online,
                          'list': public_list(lst), 'tasks': tasks_of(lst['list_id']),
                          'activity': list(reversed(lst.get('activity', [])[-15:]))})

ROUTES = {'/lists/create': handle_list_create, '/lists/join': handle_list_join,
          '/lists/mine': handle_list_mine, '/lists/delete': handle_list_delete,
          '/tasks/add': handle_task_add, '/tasks/update': handle_task_update,
          '/tasks/delete': handle_task_delete, '/sync': handle_sync}

def lambda_handler(event, context=None):
    route = ROUTES.get(event.get('path', ''))
    if route is None:
        return response(404, {'error': 'Unknown route'})
    body = event.get('body') or {}
    if isinstance(body, str):
        try:
            body = json.loads(body)
        except json.JSONDecodeError:
            return response(400, {'error': 'Body must be valid JSON.'})
    return route(body)

In [ ]:
from flask import Flask, request, Response

app = Flask(__name__)

@app.post('/api/<path:action>')
def api(action):
    res = lambda_handler({'path': '/' + action, 'body': json.dumps(request.get_json(silent=True) or {})}, None)
    return Response(res['body'], status=res['statusCode'], mimetype=res['headers']['Content-Type'])

In [ ]:
HTML_PAGE = r"""
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Collaborative To-Do</title>
<style>
  * { box-sizing: border-box; }
  body { font-family: Arial, sans-serif; background: #f1f5f9; margin: 0; color: #1f2937; }
  header { background: #0369a1; color: #fff; padding: 14px 22px; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 8px; }
  header h1 { margin: 0; font-size: 20px; }
  #who { font-size: 13px; } #live { font-size: 12px; opacity: .95; }
  .layout { display: grid; grid-template-columns: 250px 1fr 260px; gap: 16px; max-width: 1250px; margin: 16px auto; padding: 0 14px; }
  @media (max-width: 900px) { .layout { grid-template-columns: 1fr; } }
  .card { background: #fff; border-radius: 12px; padding: 14px; box-shadow: 0 2px 8px rgba(0,0,0,.08); margin-bottom: 14px; }
  h3 { margin: 0 0 10px; font-size: 15px; }
  input[type=text], input[type=date], select { padding: 7px; border: 1px solid #cbd5e1; border-radius: 8px; font-size: 13px; }
  button { background: #0369a1; color: #fff; border: none; padding: 7px 13px; border-radius: 8px; font-size: 13px; cursor: pointer; }
  button:hover { background: #075985; }
  button.danger { background: #dc2626; padding: 4px 9px; font-size: 12px; }
  button.ghost { background: #e2e8f0; color: #111; padding: 4px 9px; font-size: 12px; }
  .list-item { padding: 8px 10px; border-radius: 8px; cursor: pointer; margin-bottom: 4px; font-size: 14px; }
  .list-item:hover { background: #e0f2fe; } .list-item.active { background: #bae6fd; font-weight: bold; }
  .row { display: flex; gap: 8px; margin-bottom: 8px; flex-wrap: wrap; }
  .row input[type=text] { flex: 1; min-width: 90px; }
  .small { font-size: 12px; color: #64748b; }
  .chip { display: inline-flex; align-items: center; gap: 5px; background: #f1f5f9; border-radius: 14px; padding: 3px 10px 3px 4px; font-size: 12px; margin: 2px; }
  .av { width: 20px; height: 20px; border-radius: 50%; color: #fff; font-size: 11px; display: inline-flex; align-items: center; justify-content: center; font-weight: bold; }
  .dot { width: 8px; height: 8px; border-radius: 50%; background: #22c55e; display: inline-block; }
  .code { font-family: monospace; font-size: 18px; letter-spacing: 3px; background: #fef3c7; padding: 2px 10px; border-radius: 6px; }
  .task { display: flex; align-items: center; gap: 8px; padding: 9px 4px; border-bottom: 1px solid #e5e7eb; flex-wrap: wrap; }
  .task .title { flex: 1; min-width: 140px; font-size: 14px; cursor: text; }
  .task.done .title { text-decoration: line-through; color: #94a3b8; }
  .task input[type=text] { flex: 1; }
  .pri { font-size: 11px; font-weight: bold; padding: 2px 8px; border-radius: 10px; color: #fff; }
  .pri.High { background: #dc2626; } .pri.Medium { background: #d97706; } .pri.Low { background: #64748b; }
  .overdue { color: #dc2626; font-weight: bold; }
  .filters button { background: #e2e8f0; color: #111; margin-right: 4px; } .filters button.on { background: #0369a1; color: #fff; }
  .act { font-size: 12px; padding: 5px 0; border-bottom: 1px solid #f1f5f9; }
  #toast { position: fixed; bottom: 18px; left: 50%; transform: translateX(-50%); background: #1e293b; color: #fff; padding: 9px 16px; border-radius: 8px; font-size: 13px; display: none; z-index: 10; }
  #login { max-width: 360px; margin: 80px auto; text-align: center; }
  #login input { width: 100%; margin: 12px 0; padding: 10px; font-size: 15px; }
</style>
</head>
<body>

<div id="login" class="card">
  <h2>Collaborative To-Do</h2>
  <p class="small">Enter your name to start. Open this page in another tab with a different name to see real-time sync.</p>
  <input type="text" id="name" placeholder="Your name (letters, numbers, _)" maxlength="20">
  <button onclick="login()">Continue</button>
  <p id="loginerr" class="small" style="color:#dc2626"></p>
</div>

<div id="app" style="display:none">
<header>
  <h1>Collaborative To-Do List</h1>
  <div><span id="who"></span> &nbsp; <span id="live">connecting...</span> &nbsp; <button class="ghost" onclick="logout()">Switch user</button></div>
</header>

<div class="layout">
  <div>
    <div class="card"><h3>My lists</h3><div id="lists"></div><div id="nolist" class="small">No lists yet.</div></div>
    <div class="card"><h3>New list</h3>
      <div class="row"><input type="text" id="newtitle" placeholder="List title"><button onclick="createList()">Create</button></div>
      <h3 style="margin-top:12px">Join a list</h3>
      <div class="row"><input type="text" id="joincode" placeholder="Share code" maxlength="6"><button onclick="joinList()">Join</button></div>
      <div id="sidemsg" class="small"></div>
    </div>
  </div>

  <div>
    <div id="empty" class="card">Create a list or join one with a share code to begin.</div>
    <div id="main" style="display:none">
      <div class="card">
        <div class="row" style="justify-content:space-between;align-items:center">
          <h3 id="ltitle" style="margin:0;font-size:18px"></h3>
          <span>Share code: <span class="code" id="lcode"></span></span>
        </div>
        <div id="online" style="margin-top:6px"></div>
      </div>
      <div class="card">
        <div class="row">
          <input type="text" id="ttitle" placeholder="Add a task..." onkeydown="if(event.key==='Enter')addTask()">
          <select id="tpri"><option>Medium</option><option>High</option><option>Low</option></select>
          <select id="tas"></select>
          <input type="date" id="tdue">
          <button onclick="addTask()">Add</button>
        </div>
        <div class="filters row" id="filters"></div>
        <div id="tasks"></div>
        <div class="row" style="margin-top:10px"><span class="small" id="count"></span><button class="danger" id="dellist" style="display:none" onclick="deleteList()">Delete list</button></div>
      </div>
    </div>
  </div>

  <div>
    <div class="card"><h3>Members</h3><div id="members" class="small"></div></div>
    <div class="card"><h3>Live activity</h3><div id="activity"></div></div>
  </div>
</div>
</div>

<div id="toast"></div>

<script>
const $ = id => document.getElementById(id);
let user = null, listId = null, version = -1, tasks = [], members = [], owner = '', filter = 'all', editing = null, pollBusy = false, listsTick = 0;

function el(tag, attrs, ...kids) {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(attrs || {})) {
    if (k === 'class') e.className = v; else if (k === 'style') e.style.cssText = v;
    else if (k.startsWith('on') || k === 'checked' || k === 'value' || k === 'disabled' || k === 'selected') e[k] = v;
    else e.setAttribute(k, v);
  }
  kids.forEach(c => { if (c !== null && c !== undefined) e.append(c); });
  return e;
}
function toast(msg) { const t = $('toast'); t.textContent = msg; t.style.display = 'block'; clearTimeout(t._h); t._h = setTimeout(() => t.style.display = 'none', 3500); }
function color(name) { let h = 0; for (const c of name) h = (h * 31 + c.charCodeAt(0)) % 360; return 'hsl(' + h + ',60%,45%)'; }
function avatar(name) { return el('span', {class: 'av', style: 'background:' + color(name)}, name[0].toUpperCase()); }
async function api(action, body) {
  const r = await fetch('api/' + action, {method: 'POST', headers: {'Content-Type': 'application/json'}, body: JSON.stringify(Object.assign({user: user}, body))});
  let data = {}; try { data = await r.json(); } catch (e) {}
  return {ok: r.ok, status: r.status, data: data};
}

// ---------- login ----------
function login() {
  const n = $('name').value.trim();
  if (!/^[A-Za-z0-9_]{2,20}$/.test(n)) { $('loginerr').textContent = 'Use 2-20 letters, numbers or underscore.'; return; }
  user = n; try { sessionStorage.setItem('todo_user', n); } catch (e) {}
  start();
}
function logout() { try { sessionStorage.removeItem('todo_user'); } catch (e) {} location.reload(); }
function start() {
  $('login').style.display = 'none'; $('app').style.display = 'block';
  $('who').replaceChildren(avatar(user), ' ' + user);
  refreshLists(); setInterval(poll, 2000);
}
$('name').onkeydown = e => { if (e.key === 'Enter') login(); };
try { const s = sessionStorage.getItem('todo_user'); if (s) { user = s; start(); } } catch (e) {}

// ---------- lists ----------
async function refreshLists(selectId) {
  const r = await api('lists/mine', {}); if (!r.ok) return;
  const box = $('lists'); box.replaceChildren();
  r.data.lists.forEach(l => box.append(el('div', {class: 'list-item' + (l.list_id === listId ? ' active' : ''), onclick: () => openList(l.list_id)}, l.title + ' (' + l.members.length + ')')));
  $('nolist').style.display = r.data.lists.length ? 'none' : 'block';
  if (selectId) openList(selectId);
  else if (!listId && r.data.lists.length) openList(r.data.lists[0].list_id);
}
function openList(id) { listId = id; version = -1; $('empty').style.display = 'none'; $('main').style.display = 'block'; poll(true); refreshLists(); }
async function createList() {
  const r = await api('lists/create', {title: $('newtitle').value});
  if (!r.ok) { $('sidemsg').textContent = r.data.error; return; }
  $('newtitle').value = ''; $('sidemsg').textContent = 'List created. Share its code with friends.'; refreshLists(r.data.list_id);
}
async function joinList() {
  const r = await api('lists/join', {code: $('joincode').value});
  if (!r.ok) { $('sidemsg').textContent = r.data.error; return; }
  $('joincode').value = ''; $('sidemsg').textContent = 'Joined "' + r.data.title + '".'; refreshLists(r.data.list_id);
}
async function deleteList() {
  if (!$('dellist').dataset.sure) { $('dellist').dataset.sure = 1; $('dellist').textContent = 'Click again to confirm'; setTimeout(() => { delete $('dellist').dataset.sure; $('dellist').textContent = 'Delete list'; }, 3000); return; }
  const r = await api('lists/delete', {list_id: listId});
  if (r.ok) { listId = null; $('main').style.display = 'none'; $('empty').style.display = 'block'; refreshLists(); } else toast(r.data.error);
}

// ---------- real-time sync (polling with versions) ----------
async function poll(force) {
  if (!listId || pollBusy) return;
  pollBusy = true;
  try {
    const r = await api('sync', {list_id: listId, since: force ? -1 : version});
    if (!r.ok) {
      $('live').textContent = '○ ' + (r.data.error || 'offline');
      if (r.status === 404 || r.status === 403) { toast(r.data.error); listId = null; $('main').style.display = 'none'; $('empty').style.display = 'block'; refreshLists(); }
      return;
    }
    $('live').textContent = '● Live - synced ' + new Date().toLocaleTimeString();
    renderOnline(r.data.online);
    if (r.data.changed && !editing) {
      version = r.data.version; tasks = r.data.tasks; members = r.data.list.members; owner = r.data.list.owner;
      $('ltitle').textContent = r.data.list.title; $('lcode').textContent = r.data.list.share_code;
      $('dellist').style.display = owner === user ? 'inline-block' : 'none';
      renderMembers(r.data.online); renderAssignees(); renderTasks(); renderActivity(r.data.activity);
    }
    if (++listsTick % 5 === 0) refreshLists();
  } finally { pollBusy = false; }
}
function renderOnline(online) {
  const box = $('online'); box.replaceChildren(el('span', {class: 'small'}, 'Online now: '));
  online.forEach(n => box.append(el('span', {class: 'chip'}, avatar(n), n, el('span', {class: 'dot'}))));
  renderMembers(online);
}
function renderMembers(online) {
  const box = $('members'); box.replaceChildren();
  members.forEach(m => box.append(el('div', {style: 'margin:4px 0'}, avatar(m), ' ' + m + (m === owner ? ' (owner)' : '') + (online.includes(m) ? ' \u25CF' : ''))));
}
function renderAssignees() {
  const s = $('tas'), cur = s.value; s.replaceChildren(el('option', {value: ''}, 'Unassigned'));
  members.forEach(m => s.append(el('option', {value: m}, m))); s.value = cur;
}
function renderActivity(items) {
  const box = $('activity'); box.replaceChildren();
  items.forEach(a => box.append(el('div', {class: 'act'}, el('b', {}, a.user), ' ' + a.msg, el('div', {class: 'small'}, a.t.slice(11, 19) + ' UTC'))));
}

// ---------- tasks ----------
const FILTERS = [['all', 'All'], ['open', 'Open'], ['done', 'Done'], ['mine', 'Assigned to me']];
function renderTasks() {
  const fb = $('filters'); fb.replaceChildren();
  FILTERS.forEach(([k, label]) => fb.append(el('button', {class: filter === k ? 'on' : '', onclick: () => { filter = k; renderTasks(); }}, label)));
  const shown = tasks.filter(t => filter === 'all' || (filter === 'open' && !t.done) || (filter === 'done' && t.done) || (filter === 'mine' && t.assignee === user));
  const box = $('tasks'); box.replaceChildren();
  const today = new Date().toISOString().slice(0, 10);
  shown.forEach(t => {
    const row = el('div', {class: 'task' + (t.done ? ' done' : '')});
    row.append(el('input', {type: 'checkbox', checked: !!t.done, onchange: e => update(t, {done: e.target.checked})}));
    const title = el('span', {class: 'title', title: 'Double-click to edit', ondblclick: () => editTitle(t, title)}, t.title);
    row.append(title, el('span', {class: 'pri ' + t.priority}, t.priority));
    const sel = el('select', {onchange: e => update(t, {assignee: e.target.value})}, el('option', {value: ''}, 'Unassigned'));
    members.forEach(m => sel.append(el('option', {value: m}, m))); sel.value = t.assignee || '';
    row.append(sel);
    if (t.due_date) row.append(el('span', {class: 'small' + (!t.done && t.due_date < today ? ' overdue' : '')}, 'due ' + t.due_date));
    row.append(el('span', {class: 'small'}, 'by ' + t.updated_by));
    const del = el('button', {class: 'danger', onclick: () => removeTask(t, del)}, 'Delete');
    row.append(del); box.append(row);
  });
  if (!shown.length) box.append(el('div', {class: 'small', style: 'padding:10px'}, 'No tasks here.'));
  $('count').textContent = tasks.filter(t => !t.done).length + ' open, ' + tasks.filter(t => t.done).length + ' done';
}
function editTitle(t, span) {
  editing = t.task_id;
  const input = el('input', {type: 'text', value: t.title, maxlength: 120});
  span.replaceWith(input); input.focus(); input.select();
  let finished = false;
  const finish = async save => {
    if (finished) return; finished = true; editing = null;
    if (save && input.value.trim() && input.value.trim() !== t.title) await update(t, {title: input.value.trim()}); else poll(true);
  };
  input.onkeydown = e => { if (e.key === 'Enter') finish(true); if (e.key === 'Escape') finish(false); };
  input.onblur = () => finish(true);
}
async function addTask() {
  const title = $('ttitle').value; if (!title.trim()) return;
  const r = await api('tasks/add', {list_id: listId, title: title, priority: $('tpri').value, assignee: $('tas').value, due_date: $('tdue').value});
  if (!r.ok) { toast(r.data.error); return; }
  $('ttitle').value = ''; $('tdue').value = ''; poll(true);
}
async function update(t, fields) {
  const r = await api('tasks/update', Object.assign({list_id: listId, task_id: t.task_id, version: t.version}, fields));
  if (!r.ok) toast(r.data.error);
  poll(true);
}
async function removeTask(t, btn) {
  if (!btn.dataset.sure) { btn.dataset.sure = 1; btn.textContent = 'Sure?'; setTimeout(() => { btn.dataset.sure = ''; btn.textContent = 'Delete'; }, 2500); return; }
  const r = await api('tasks/delete', {list_id: listId, task_id: t.task_id});
  if (!r.ok) toast(r.data.error);
  poll(true);
}
</script>
</body>
</html>
"""

@app.get('/')
def home():
    return Response(HTML_PAGE, mimetype='text/html')

In [ ]:
client = app.test_client()

def call(action, **body):
    r = client.post('/api/' + action, json=body)
    return r.status_code, r.get_json()

s, lst = call('lists/create', user='Alice', title='Group Project')
print('Alice created list:', lst['title'], '| share code:', lst['share_code'])

s, joined = call('lists/join', user='Bob', code=lst['share_code'])
print('Bob joined. Members:', joined['members'])

lid = lst['list_id']
_, t1 = call('tasks/add', user='Alice', list_id=lid, title='Write report', priority='High', assignee='Bob')
_, t2 = call('tasks/add', user='Bob',   list_id=lid, title='Prepare slides', priority='Medium', assignee='Alice')
_, t3 = call('tasks/add', user='Alice', list_id=lid, title='Book meeting room', priority='Low')
print('Tasks added:', [t1['title'], t2['title'], t3['title']])

# Real-time sync: Bob polls with the version he already has
_, snap = call('sync', user='Bob', list_id=lid, since=-1)
v = snap['version']
print(f'\nBob first sync: changed={snap["changed"]}, version={v}, tasks={len(snap["tasks"])}')
print('Bob polls again (nothing new):', call('sync', user='Bob', list_id=lid, since=v)[1]['changed'])
call('tasks/update', user='Alice', list_id=lid, task_id=t3['task_id'], version=t3['version'], done=True)
print('After Alice completes a task, Bob polls:', call('sync', user='Bob', list_id=lid, since=v)[1]['changed'])

# Conflict: both edit the SAME task version
s_a, _ = call('tasks/update', user='Alice', list_id=lid, task_id=t1['task_id'], version=t1['version'], priority='Medium')
s_b, c = call('tasks/update', user='Bob',   list_id=lid, task_id=t1['task_id'], version=t1['version'], title='Write final report')
print(f'\nAlice update status: {s_a} | Bob update (stale version) status: {s_b} -> {c["error"]}')

_, final = call('sync', user='Alice', list_id=lid, since=-1)
print('\nActivity feed:')
for a in final['activity']:
    print(f"  {a['t'][11:19]}  {a['user']:6} {a['msg']}")

Alice created list: Group Project | share code: 66RAE9
Bob joined. Members: ['Alice', 'Bob']
Tasks added: ['Write report', 'Prepare slides', 'Book meeting room']

Bob first sync: changed=True, version=6, tasks=3
Bob polls again (nothing new): False
After Alice completes a task, Bob polls: True

Alice update status: 200 | Bob update (stale version) status: 409 -> Someone else changed this task first. Showing the latest version.

Activity feed:
  11:25:40  Alice  set priority Medium on "Write report"
  11:25:40  Alice  completed "Book meeting room"
  11:25:40  Alice  added "Book meeting room"
  11:25:40  Bob    added "Prepare slides"
  11:25:40  Alice  added "Write report"
  11:25:40  Bob    joined the list
  11:25:40  Alice  created the list "Group Project"


In [ ]:
from werkzeug.serving import make_server

try:                      # allow re-running this cell
    server.shutdown()
except Exception:
    pass

server = make_server('0.0.0.0', 5000, app, threaded=True)
threading.Thread(target=server.serve_forever, daemon=True).start()
print('Flask server running on port 5000')

try:
    from google.colab import output
    output.serve_kernel_port_as_iframe(5000, height=1000)    # shows the website below this cell
    # output.serve_kernel_port_as_window(5000)               # BEST for the demo: opens a new browser tab. Run it twice
    #                                                        # (two tabs), sign in as different users and watch them sync.
except Exception:
    print('Open http://127.0.0.1:5000 in your browser')

Flask server running on port 5000


<IPython.core.display.Javascript object>